# ACF der simulierten Datensaetze (alle vier Szenarien)

Berechnung, Fit, Binning und Plot der ACF der Groesse `X` fuer alle vier simulierten
Szenarien aus `data/simulated/`. Die eigentliche Logik liegt zentral im Modul
**`acf_analysis`** (`src/acf_analysis/acf.py`) und wird hier nur importiert und aufgerufen —
derselbe Code laeuft unveraendert auf simulierten und echten Daten.

**Welcher Fit pro Szenario berichtet wird** (Konfiguration im Setup, Feld `primary`):
freies `tau` nur beim AR(1)-Szenario, fixes `tau = 7` bei den drei uebrigen. Das Modul
rechnet technisch immer beide Varianten; berichtet und geplottet wird nur die festgelegte.

**Zur Darstellung:** Die Fit-Kurve ist in allen vier Panels rot; die Grafik enthaelt weder
Legende noch Fit-Parameter. Welcher Fit gezeigt wird und mit welchen Werten, steht im
Kurzbericht (Zelle 1) und in der Ergebnistabelle (Zelle 2).

## Setup — Modul, Datensaetze, Konfiguration

Die vier Szenarien sind unten als Konfigurationsliste hinterlegt: je Datensatz Dateiname,
Anzeigename, welcher Fit *primaer* ist (`free` nur bei AR(1)) und der AR(1)-Parameter
`alpha` fuer den Soll-`tau`-Vergleich.

Damit das Modul `acf_analysis` auch ohne `pip install -e .` importierbar ist, sucht das
Notebook die Projektwurzel (Ordner mit `pyproject.toml`) und legt deren `src/` auf den
Importpfad. Der Datenpfad wird aus derselben Wurzel abgeleitet.

**Achsenbeschriftungen** stehen als `XLABEL` / `YLABEL` / `POINT_LABEL` im Setup und werden
`plot_acf` explizit uebergeben — die Modul-Defaults entsprechen dem Fussballfall, das
Volleyball-Notebook ueberschreibt sie.

In [ ]:
import sys                                            # fuer sys.path (Import-Bootstrap)
from pathlib import Path                                # Pfade
import numpy as np                                      # numerische Arrays
import pandas as pd                                     # Datentabellen
import matplotlib.pyplot as plt                         # Plots

# --- acf_analysis reproduzierbar importierbar machen ----------------------
# Projektwurzel = naechster uebergeordneter Ordner mit pyproject.toml; dessen src/ auf den
# Importpfad legen. So laeuft das Notebook fuer jeden Klon -- mit ODER ohne `pip install -e .`
# -- und unabhaengig davon, wie tief das Notebook im notebooks/-Baum liegt.
ROOT = Path.cwd()                                       # Startpunkt: Verzeichnis des Notebooks
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent                                 # nach oben laufen, bis pyproject.toml gefunden
if str(ROOT / "src") not in sys.path:                  # src/ nur einmal einhaengen
    sys.path.insert(0, str(ROOT / "src"))

from acf_analysis import compute_acf, fit_acf, bin_acf, plot_acf  # zentrale ACF-Logik (Modul)

# --- Pfad zu den simulierten long_df-Datensaetzen (aus derselben Wurzel) ---
DATA_DIR = ROOT / "data" / "simulated"

# --- Achsenbeschriftungen (sportartabhaengig, deshalb hier und nicht im Modul) ---
# plot_acf nimmt xlabel/ylabel/point_label als Parameter; die Modul-Defaults passen zum
# Fussball (Spieltage, Tordifferenz). Die simulierten Daten bilden einen Fussball-Spielplan
# ab, also bleiben sie hier unveraendert -- im Volleyball-Notebook werden sie ueberschrieben
# (Delta m zaehlt dort PARTIEN, X ist eine Punktedifferenz).
XLABEL      = r"$\Delta m$"            # x-Achse: Lag in Spieltagen
YLABEL      = r"$K(\Delta m)$"                          # y-Achse: ACF-Wert in Absolutwerten
POINT_LABEL = "ACF (gebinnt, nur Darstellung)"          # Legendeneintrag der Datenpunkte

# --- Gitterform der Vergleichsplots (Zelle 3, 4, 5: je vier Panels) ------
# Steuert zugleich, WO die Achsenbeschriftung steht: xlabel nur in der untersten Reihe,
# ylabel nur in der linken Spalte -- innen waere der Achsentitel Wiederholung und wuerde
# den Panels Platz wegnehmen.
NROWS, NCOLS = 2, 2

# --- Sammelstelle fuer die erzeugten Abbildungen (Export-Zelle am Ende) ---
figures = {}                                            # name -> Figure

# --- Normierung: hier bewusst AUS -----------------------------------------
# plot_acf teilt standardmaessig durch a+b = K_fit(0) (CLAUDE.md Abschnitt 6). Auf den
# simulierten Daten sind die ABSOLUTWERTE die Aussage: sie werden gegen die bekannten
# Sollwerte der Simulation geprueft (z.B. K ~ Var[S] beim konstanten Szenario, K ~ 0 bei
# zufaelliger Tagesform). Eine Normierung wuerde genau diesen Vergleich zerstoeren --
# und beim Szenario "Zufaellige Tagesform" waere sie mit a+b ~ 0 ohnehin undefiniert.
NORMALIZE = False

# --- Konfiguration der vier Szenarien -------------------------------------
# primary: welcher Fit fuer dieses Szenario die Hauptaussage ist
#   "free"  -> freie Abklingzeit tau (nur AR(1): echter exponentieller Abfall)
#   "tau7"  -> fixes tau=7 (Literaturwert), weil tau hier nicht identifizierbar ist
DATASETS = [
    {"file": "constant_31seasons.csv",     "label": "Konstante Leistungsstärke",
     "primary": "tau7", "alpha": None,
     "note": "Konstante Leistungsstaerke -> flache ACF, keine Formphase (b ~ 0). "
             "Die abklingende Komponente fehlt, tau ist nicht identifizierbar; Fit mit tau=7."},
    {"file": "random_31seasons.csv",       "label": "Zufällige Leistungsstärke",
     "primary": "tau7", "alpha": None,
     "note": "Unkorrelierte Tagesform -> K ~ 0 fuer Delta m >= 1 (b ~ 0). "
             "tau nicht identifizierbar; Fit mit tau=7."},
    {"file": "const_form_31seasons.csv",   "label": "Basisstärke + Tagesform",
     "primary": "tau7", "alpha": None,
     "note": "Weitgehend flache ACF; der freie Fit laeuft an die tau-Grenze (schlecht "
             "bestimmt). Fit mit tau=7 als Referenz."},
    {"file": "ar1_31seasons_alpha0.90.csv", "label": "AR(1)-Prozess, α = 0,90",
     "primary": "free", "alpha": 0.90,
     "note": "Exponentieller Abfall -> tau frei gefittet und mit dem Sollwert "
             "tau_soll = -1/ln(alpha) verglichen."},
]

# --- Schema-/Invarianten-Check (bricht ab, falls Aufbereitung fehlt) ------
REQUIRED = ["season_id", "date", "team_id", "opponent_id", "X", "is_home", "match_number"]  # Pflichtspalten

def check_long_df(df):
    """Prueft die long_df-Invarianten aus CLAUDE.md, bevor die ACF gerechnet wird."""
    assert all(c in df.columns for c in REQUIRED), "long_df fehlen Spalten"                  # Schema vollstaendig?
    for _, g in df.groupby(["season_id", "team_id"], sort=False):                            # jede Team-Saison
        assert np.array_equal(g["match_number"].to_numpy(), np.arange(len(g))), "nicht chronologisch"  # 0..n-1
    assert df.groupby("season_id")["X"].mean().abs().max() < 1e-9, "E[X] != 0 pro Saison"    # E[X]=0 per Konstruktion

print("Modul geladen, Konfiguration fuer", len(DATASETS), "Szenarien bereit.")
print(f"Normierung auf a+b: {NORMALIZE}  (Absolutwerte, damit die Sollwerte pruefbar bleiben)")
for dataset in DATASETS:                                # kurze Uebersicht
    print(f"  - {dataset['label']:34s} (primaerer Fit: {dataset['primary']})")

## Zelle 1 — Batch: ACF, Fit und Binning für alle Szenarien

Über alle vier Datensätze schleifen und je Datensatz `compute_acf → fit_acf → bin_acf`
aufrufen (alles aus dem Modul). Ergebnisse werden in `results` gesammelt und weiter unten
für Tabelle und Plot verwendet.

`fit_acf` liefert beide Varianten zusammen; **ausgegeben wird nur die für das Szenario
festgelegte** — `τ = 7` (fix) bei den ersten drei, freies `τ` beim AR(1). Beim AR(1) wird
`τ` zusätzlich mit dem Sollwert `τ = −1/ln α` verglichen.

In [ ]:
results = {}                                             # label -> dict(dataset, acf, fit, bins); label des datasets fungiert als Key

# TODO: AR(1) mit alpha=0.9, 0.8, 0.7, 0.6, 0.5 simulieren und vergleichen --> separat plotten!

for dataset in DATASETS:
    long_df = pd.read_csv(DATA_DIR / dataset["file"])
    check_long_df(long_df)

    acf  = compute_acf(long_df)                         # ACF K(Delta m), N, S, SQ, sigma
    fit  = fit_acf(acf)                                 # Modul rechnet beide Varianten ...
    bins = bin_acf(acf, n_min=9000)
    results[dataset["label"]] = {"dataset": dataset, "acf": acf, "fit": fit, "bins": bins}

    # --- Kurzbericht: NUR der fuer dieses Szenario festgelegte Fit ----------
    # ... berichtet wird hier aber ausschliesslich die Variante aus dataset["primary"]:
    #   "free" -> freies tau (nur AR(1), dort ist tau identifizierbar)
    #   "tau7" -> fixes tau = 7 (Literaturwert), da ohne Abfall nicht bestimmbar
    print(f"### {dataset['label']}  ({dataset['file']})")
    print(f"    Lags: {len(acf)},  K(1) = {acf['K'].iloc[0]:.3f},  K(letzter) = {acf['K'].iloc[-1]:.3f}")
    if dataset["primary"] == "free":                    # freier Fit: a, b, tau
        tau_soll = -1.0 / np.log(dataset["alpha"])      # Sollwert aus dem AR(1)-Parameter
        print( "    Fit (freies tau):")
        print(f"        a        = {fit.a:+.4f} +- {fit.a_err:.4f}")
        print(f"        b        = {fit.b:+.4f} +- {fit.b_err:.4f}")
        print(f"        tau      = {fit.tau:.2f} +- {fit.tau_err:.2f}"
              f"   (Soll: -1/ln({dataset['alpha']}) = {tau_soll:.2f})")
        print(f"        chi2/dof = {fit.chi2_red:.2f}   corr(b,tau) = {fit.corr_b_tau:+.2f}")
    else:                                               # fixes tau = 7: nur a, b frei
        print( "    Fit (tau = 7 fix):")
        print(f"        a        = {fit.a7:+.4f} +- {fit.a7_err:.4f}")
        print(f"        b        = {fit.b7:+.4f} +- {fit.b7_err:.4f}")
        print(f"        chi2/dof = {fit.chi2_red7:.2f}")
    print(f"    {dataset['note']}")
    print()

## Zelle 2 — Ergebnistabelle

Je Szenario **eine** Zeile mit den Parametern des dafür festgelegten Fits: `τ = 7` (fix) bei
den ersten drei Szenarien, freies `τ` beim AR(1). Die Spalte `Fit` hält fest, welche Variante
die Zeile zeigt; `τ ±` und `τ Soll` bleiben bei den fixen Fits leer, weil `τ` dort kein
Schätzwert ist. `a` und `b` der drei `τ = 7`-Zeilen sind untereinander direkt vergleichbar
(gleiche Modellform).

In [ ]:
rows = []
for label, r in results.items():
    dataset, fit = r["dataset"], r["fit"]
    if dataset["primary"] == "free":                    # AR(1): freies tau, mit Sollwert
        tau_soll = -1.0 / np.log(dataset["alpha"])
        row = {"Fit": "tau frei",
               "a": fit.a,   "a +-": fit.a_err,
               "b": fit.b,   "b +-": fit.b_err,
               "tau": fit.tau, "tau +-": fit.tau_err, "tau Soll": tau_soll,
               "chi2/dof": fit.chi2_red}
    else:                                               # tau = 7 fix: nur a, b geschaetzt
        row = {"Fit": "tau = 7 (fix)",
               "a": fit.a7,  "a +-": fit.a7_err,
               "b": fit.b7,  "b +-": fit.b7_err,
               "tau": 7.0,   "tau +-": np.nan, "tau Soll": np.nan,   # tau ist kein Schaetzwert
               "chi2/dof": fit.chi2_red7}
    rows.append({"Szenario": label, **row})

table = pd.DataFrame(rows).round({"a": 4, "a +-": 4, "b": 4, "b +-": 4,
                                  "tau": 2, "tau +-": 2, "tau Soll": 2, "chi2/dof": 2})
print(table.to_string(index=False, na_rep="-"))

# TODO: Hier raus --> Gehört in die Diskussion.
print("\nAnmerkungen zur Abklingzeit tau je Szenario:")
for label, r in results.items():                        # Begruendung, warum tau (nicht) frei gefittet wird
    print(f"  - {label}: {r['dataset']['note']}")

## Zelle 3 — Vergleichsplot (2×2)

Gebinnte ACF-Punkte mit Fehlerbalken je Szenario, darüber **genau eine** Fit-Kurve (rot):
`τ = 7` (fix) bei den ersten drei Szenarien, freies `τ` beim AR(1). Die jeweils andere
Variante wird nicht mitgezeichnet (`show_other=False`), da sie für das Szenario nicht die
gefittete Aussage ist.

Die Grafik enthält **keine Legende und keine Fit-Parameter** (`legend=False`) — welcher Fit
gezeigt wird und mit welchen Werten, steht im Kurzbericht (Zelle 1) und in der
Ergebnistabelle (Zelle 2).

> **Wichtig:** Das Binning dient **nur der Darstellung** und geht **nicht** in den Fit ein.
> Die Fit-Kurven stammen aus dem Fit auf allen individuellen `Δm`.

**Achsenbeschriftung nur außen:** `Δm` steht nur unter der unteren Reihe, `K(Δm)` nur links
neben der linken Spalte — innen wäre der Achsentitel eine Wiederholung und würde den Panels
Platz wegnehmen (analog `acf_vb_simple.ipynb`). Die **y-Achsen bleiben ungeteilt**: da hier
nicht normiert wird (`NORMALIZE = False`, Setup), liegen die Absolutwerte der vier Szenarien
um Größenordnungen auseinander, und eine gemeinsame Skala würde die flachen Szenarien
plattdrücken. Jedes Panel behält deshalb seine eigenen Ticks. Das ist kein Widerspruch zum
gemeinsamen Achsentitel: der benennt die **Größe** `K(Δm)` (Einheit Tore², in allen vier
Panels dieselbe), den **Maßstab** lesen die Ticks — und die stehen weiterhin an jedem Panel.


In [ ]:
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(13, 9))  # Gitterform aus dem Setup
for i, (label, r) in enumerate(results.items()):        # je Szenario ein Subplot
    row, col = divmod(i, NCOLS)                         # Position im Gitter
    # Achsenbeschriftung nur AUSSEN (untere Reihe / linke Spalte): innen ist der
    # Achsentitel redundant und kostet nur Platz, der den Panels fehlt.
    # Die y-Achsen bleiben dabei bewusst UNGETEILT -- jedes Panel behaelt seine eigenen
    # Ticks. Das ist hier wichtig, weil NORMALIZE=False ist und die Absolutwerte je
    # Szenario um Groessenordnungen auseinanderliegen (K ~ Var[S] beim konstanten
    # Szenario, K ~ 0 bei zufaelliger Tagesform). Der weggelassene Achsentitel aendert
    # daran nichts: er benennt die GROESSE K(Delta m), nicht ihren Massstab -- den lesen
    # die Ticks, und die stehen weiterhin an jedem Panel.
    plot_acf(r["bins"], r["fit"], primary=r["dataset"]["primary"],  # Modul-Plotfunktion
             ax=axes[row, col], title=label, normalize=NORMALIZE,   # nur der festgelegte Fit
             point_label=POINT_LABEL,
             xlabel=XLABEL if row == NROWS - 1 else None,           # nur untere Reihe
             ylabel=YLABEL if col == 0 else None)                   # nur linke Spalte
#fig.suptitle("ACF der bereinigten Tordifferenz — simulierte Szenarien", fontsize=14)
fig.tight_layout()                                      # Layout aufraeumen
figures["simulated_acf_31seasons"] = fig                 # fuer die Export-Zelle am Ende
plt.show()

## Zelle 4 — Dieselben vier Szenarien mit 300 Saisons

Gegenprobe mit den grossen Laeufen aus `data/simulated/*_300seasons*.csv`: identische
Simulationsparameter, nur rund zehnmal so viele Saisons. Alles andere bleibt gleich —
derselbe Spielplan, dieselbe ACF-Logik, dieselbe Binning-Schwelle `n_min = 9000` und
derselbe pro Szenario festgelegte Fit.

Bei 300 Saisons liegt praktisch jeder einzelne Lag ueber `n_min`, es wird also kaum noch
etwas zusammengefasst (37 Lags -> 36 Bins). Die Tabelle darunter stellt `a` (und beim AR(1)
`tau`) beider Laeufe nebeneinander.

In [ ]:
N_MIN = 9000                                            # absolute Schwelle, wie oben (nicht skalieren)

# --- dieselbe Konfiguration, nur die 300-Saisons-Dateien ------------------
# Label, primaerer Fit und alpha werden unveraendert aus DATASETS uebernommen; getauscht
# wird nur der Dateiname, damit beide Laeufe garantiert dieselbe Fit-Vorgabe verwenden.
DATASETS_300 = [{**dataset, "file": dataset["file"].replace("31seasons", "300seasons")}
                for dataset in DATASETS]
for dataset in DATASETS_300:
    assert (DATA_DIR / dataset["file"]).exists(), f"Datei fehlt: {dataset['file']}"


def run_scenarios(datasets, n_min=N_MIN):
    """ACF, Fit und Binning fuer eine Liste von Szenarien; berichtet nur den festgelegten Fit.

    Gleicher Ablauf wie in Zelle 1 (compute_acf -> fit_acf -> bin_acf), hier als Funktion,
    damit der 300-Saisons-Lauf denselben Codepfad nimmt wie der 31-Saisons-Lauf.
    """
    out = {}
    for dataset in datasets:
        long_df = pd.read_csv(DATA_DIR / dataset["file"])
        check_long_df(long_df)                          # long_df-Invarianten (CLAUDE.md)

        acf  = compute_acf(long_df)                     # ACF K(Delta m), N, S, SQ, sigma
        fit  = fit_acf(acf)                             # Modul rechnet beide Varianten ...
        bins = bin_acf(acf, n_min=n_min)                # Binning NUR fuer die Darstellung
        out[dataset["label"]] = {"dataset": dataset, "acf": acf, "fit": fit, "bins": bins}

        # --- Kurzbericht: NUR die in dataset["primary"] festgelegte Variante
        print(f"### {dataset['label']}  ({dataset['file']})")
        print(f"    Saisons: {long_df['season_id'].nunique()},  "
              f"Team-Saisons: {long_df.groupby(['season_id', 'team_id']).ngroups},  "
              f"Paare bei Delta m = 1: {int(acf['N'].iloc[0]):,}")
        print(f"    Lags: {len(acf)} -> {len(bins)} Bins (n_min={n_min}),  "
              f"K(1) = {acf['K'].iloc[0]:.3f},  K(letzter) = {acf['K'].iloc[-1]:.3f}")
        if dataset["primary"] == "free":                # freier Fit: a, b, tau
            tau_soll = -1.0 / np.log(dataset["alpha"])  # Sollwert aus dem AR(1)-Parameter
            print( "    Fit (freies tau):")
            print(f"        a        = {fit.a:+.4f} +- {fit.a_err:.4f}")
            print(f"        b        = {fit.b:+.4f} +- {fit.b_err:.4f}")
            print(f"        tau      = {fit.tau:.2f} +- {fit.tau_err:.2f}"
                  f"   (Soll: -1/ln({dataset['alpha']}) = {tau_soll:.2f})")
            print(f"        chi2/dof = {fit.chi2_red:.2f}   corr(b,tau) = {fit.corr_b_tau:+.2f}")
        else:                                           # fixes tau = 7: nur a, b frei
            print( "    Fit (tau = 7 fix):")
            print(f"        a        = {fit.a7:+.4f} +- {fit.a7_err:.4f}")
            print(f"        b        = {fit.b7:+.4f} +- {fit.b7_err:.4f}")
            print(f"        chi2/dof = {fit.chi2_red7:.2f}")
        print()
    return out


results_300 = run_scenarios(DATASETS_300)               # ACF + Fit + Binning, mit Kurzbericht

# --- Vergleich der Plateaus: 31 vs. 300 Saisons --------------------------
# a ist der Anteil der saisonkonstanten Leistungsstaerke; er darf sich zwischen den beiden
# Laeufen nur im Rahmen der Fehler unterscheiden -- die Simulation ist ja dieselbe.
rows = []
for label, r300 in results_300.items():
    f31, f300 = results[label]["fit"], r300["fit"]      # results = 31 Saisons (Zelle 1)
    frei = r300["dataset"]["primary"] == "free"         # welcher Fit gilt fuer dieses Szenario?
    rows.append({
        "Szenario":     label,
        "Fit":          "tau frei" if frei else "tau = 7 (fix)",
        "a (31)":       f31.a  if frei else f31.a7,     "a +- (31)":  f31.a_err  if frei else f31.a7_err,
        "a (300)":      f300.a if frei else f300.a7,    "a +- (300)": f300.a_err if frei else f300.a7_err,
        "tau (31)":     f31.tau  if frei else np.nan,   # tau nur beim freien Fit ein Schaetzwert
        "tau (300)":    f300.tau if frei else np.nan,
    })
print(pd.DataFrame(rows).round(4).to_string(index=False, na_rep="-"))

# --- Vergleichsplot (2x2), gleiche Darstellung wie Zelle 3 ---------------
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(13, 9))  # Gitterform wie in Zelle 3
for i, (label, r) in enumerate(results_300.items()):
    row, col = divmod(i, NCOLS)
    plot_acf(r["bins"], r["fit"], primary=r["dataset"]["primary"],
             ax=axes[row, col], title=label, normalize=NORMALIZE,   # nur der festgelegte Fit
             point_label=POINT_LABEL,
             xlabel=XLABEL if row == NROWS - 1 else None,           # Beschriftung nur aussen
             ylabel=YLABEL if col == 0 else None)                   # (Begruendung: Zelle 3)
#fig.suptitle("ACF der bereinigten Tordifferenz — simulierte Szenarien, 300 Saisons", fontsize=14)
fig.tight_layout()
figures["simulated_acf_300seasons"] = fig                 # fuer die Export-Zelle am Ende
plt.show()

## Zelle 5 — AR(1) mit verschiedenen `alpha` (300 Saisons)

Vier AR(1)-Datensaetze, die sich nur im Parameter `alpha` unterscheiden. `tau` wird in allen
vier Panels frei gefittet (`primary = "free"`) und in der Tabelle darunter direkt gegen den
Sollwert `tau = -1/ln(alpha)` gestellt.

Die Panels sind wie in Zelle 3/4 durchnummeriert — `plot_acf` stellt `(a)`, `(b)`, … voran,
sobald mehrere Achsen in einer Figur liegen; als Titel steht nur der `alpha`-Wert. Die
Fit-Parameter erscheinen nicht in der Grafik, sondern im Kurzbericht und in der Tabelle.

**Zum `alpha` der Datei `…alpha0.87.csv`:** der Dateiname ist auf zwei Nachkommastellen
gerundet. In der Konfiguration unten steht deshalb der *echte* Wert `exp(-1/7) = 0,8669`.
Wurde die Datei stattdessen mit glatt `0,87` erzeugt, dort einfach `0.87` eintragen.

In [ ]:
V_SIM = 0.5071                                          # Var(S) der Simulation -> Soll-Amplitude b

# --- Konfiguration: Datei + ECHTES alpha ---------------------------------
# Der Dateiname ist auf zwei Stellen gerundet; massgeblich ist der tatsaechlich simulierte
# Wert, denn er legt den Sollwert tau = -1/ln(alpha) fest.
ALPHA_FILES = [
    ("ar1_300seasons_alpha0.30.csv", 0.30),                    # tau =  0.83
    ("ar1_300seasons_alpha0.50.csv", 0.50),                    # tau =  1.44
    ("ar1_300seasons_alpha0.87.csv", float(np.exp(-1 / 7))),   # = 0.8669 -> tau = 7.00 (Literaturwert)
    ("ar1_300seasons_alpha0.97.csv", 0.97),                    # tau = 32.83
]

# label = Panel-Ueberschrift (nur der alpha-Wert; (a), (b), ... setzt plot_acf selbst davor)
# primary = "free": beim AR(1) gibt es einen echten Abfall, tau ist identifizierbar
DATASETS_ALPHA = [{"file": f, "alpha": a, "primary": "free",
                   "label": f"α = {a:.2f}".replace(".", ",")}
                  for f, a in ALPHA_FILES]
for dataset in DATASETS_ALPHA:
    assert (DATA_DIR / dataset["file"]).exists(), f"Datei fehlt: {dataset['file']}"

results_alpha = run_scenarios(DATASETS_ALPHA)           # ACF + Fit + Binning, Kurzbericht wie in Zelle 4

# --- Ergebnistabelle: gefittetes tau gegen den Sollwert ------------------
# "Abw. (sigma)" = (tau_fit - tau_soll) / sigma_tau -- Betrag <~ 2 heisst: mit dem Sollwert
# vertraeglich. Ebenso sollte b nahe v und a nahe 0 liegen.
rows = []
for i, (label, r) in enumerate(results_alpha.items()):
    fit, alpha = r["fit"], r["dataset"]["alpha"]
    tau_soll = -1.0 / np.log(alpha)                     # Sollwert aus dem AR(1)-Parameter
    rows.append({
        "Panel":        f"({chr(ord('a') + i)})",
        "alpha":        alpha,
        "a":            fit.a,    "a +-": fit.a_err,    # Soll: 0
        "b":            fit.b,    "b +-": fit.b_err,    # Soll: v
        "b Soll (v)":   V_SIM,
        "tau":          fit.tau,  "tau +-": fit.tau_err,
        "tau Soll":     tau_soll,
        "Abw. (sigma)": (fit.tau - tau_soll) / fit.tau_err,
        "chi2/dof":     fit.chi2_red,
    })
table_alpha = pd.DataFrame(rows).round({"alpha": 4, "a": 4, "a +-": 4, "b": 4, "b +-": 4,
                                        "b Soll (v)": 4, "tau": 2, "tau +-": 2, "tau Soll": 2,
                                        "Abw. (sigma)": 1, "chi2/dof": 2})
print(table_alpha.to_string(index=False))

# --- Vergleichsplot (2x2), gleiche Darstellung wie Zelle 3/4 -------------
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(13, 9))  # Gitterform wie in Zelle 3
for i, (label, r) in enumerate(results_alpha.items()):
    row, col = divmod(i, NCOLS)
    plot_acf(r["bins"], r["fit"], primary=r["dataset"]["primary"],   # freier Fit, rote Kurve
             ax=axes[row, col], title=label, normalize=NORMALIZE,    # nur der festgelegte Fit
             point_label=POINT_LABEL,
             xlabel=XLABEL if row == NROWS - 1 else None,            # Beschriftung nur aussen
             ylabel=YLABEL if col == 0 else None)                    # (Begruendung: Zelle 3)
fig.tight_layout()
figures["simulated_acf_alpha_serie"] = fig                 # fuer die Export-Zelle am Ende
plt.show()

## Zelle 6 — Export der Abbildungen (PDF fuer LaTeX)

Schreibt alle in `figures` gesammelten Abbildungen nach `figures/`. Format ist PDF.

In [ ]:
SAVE    = True                                          # auf False setzen, um nichts zu schreiben
FORMATS = ["pdf"]                                       # z.B. ["pdf", "png"] fuer eine Vorschau-Rasterversion

# Zielbreite der Abbildung in Zoll (LaTeX-Textbreite). None -> figsize unveraendert lassen.
# 6.3 Zoll ~ 16 cm, die uebliche Textbreite bei A4 mit Standardraendern. Die Hoehe wird
# proportional mitskaliert, damit das Seitenverhaeltnis erhalten bleibt.
FIGSIZE_LATEX = None                                    # z.B. 6.3

FIGURES_DIR = ROOT / "figures"

# TrueType statt Type-3: Text im PDF bleibt durchsuchbar/kopierbar (LaTeX-freundlich).
plt.rcParams["pdf.fonttype"] = 42

if SAVE:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    for name, fig in figures.items():
        if FIGSIZE_LATEX:                               # auf Textbreite bringen, Seitenverhaeltnis halten
            w, h = fig.get_size_inches()
            fig.set_size_inches(FIGSIZE_LATEX, FIGSIZE_LATEX * h / w)
            fig.tight_layout()
        for ext in FORMATS:
            path = FIGURES_DIR / f"{name}.{ext}"
            fig.savefig(path, bbox_inches="tight")      # PDF ist vektoriell -> kein dpi noetig
            w, h = fig.get_size_inches()
            print(f"geschrieben: {path.relative_to(ROOT)}  ({w:.1f} x {h:.1f} Zoll)")
else:
    print("SAVE = False -- nichts geschrieben.")